# 02 - Retrieval Baseline

Main thing to check here: whether embedding search can retrieve useful Turkish legal QA passages from the passage file created in notebook 01.

No answer generation here yet. First we just want to see whether retrieval is good enough to build on.

## Before Running

Run `01_data_exploration_preprocessing.ipynb` first so these files exist locally:

- `data/processed/passages_seed.jsonl`
- `data/evaluation/heldout_seed.jsonl`

If the imports fail, run the install cell below once.

In [46]:
# Install these if the notebook environment does not have them yet:
# %pip install sentence-transformers faiss-cpu pandas numpy tqdm

In [47]:
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import faiss
    HAVE_FAISS = True
except ImportError:
    faiss = None
    HAVE_FAISS = False

from sentence_transformers import SentenceTransformer

In [48]:
ROOT = Path("..").resolve()
PASSAGES_PATH = ROOT / "data" / "processed" / "passages_seed.jsonl"
HELDOUT_PATH = ROOT / "data" / "evaluation" / "heldout_seed.jsonl"
RESULTS_DIR = ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("PASSAGES_PATH:", PASSAGES_PATH)
print("HELDOUT_PATH:", HELDOUT_PATH)
print("FAISS available:", HAVE_FAISS)

PASSAGES_PATH: /Users/onur/Desktop/CS455/Term Project/data/processed/passages_seed.jsonl
HELDOUT_PATH: /Users/onur/Desktop/CS455/Term Project/data/evaluation/heldout_seed.jsonl
FAISS available: True


## Load Seed Files

In [49]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run notebook 01 first.")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


passages = read_jsonl(PASSAGES_PATH)
heldout = read_jsonl(HELDOUT_PATH)

print("passages:", len(passages))
print("heldout:", len(heldout))
pd.DataFrame(passages[:3])

passages: 18066
heldout: 50


,passage_id,text,title,snippet,tag,source_dataset,source_split,source_row
0,orioncaf_0,Soru: İstihkak davasının amacı nedir ve hangi ...,İstihkak davasının amacı nedir ve hangi duruml...,"İstihkak davasının amacı, haksız olarak elde b...",legal_qa,OrionCAF/turkish_law_qa_dataset,train,0
1,orioncaf_1,Soru: Tapuya kayıtlı taşınmazlarda istihkak da...,Tapuya kayıtlı taşınmazlarda istihkak davasını...,Tapuya kayıtlı taşınmazlarda istihkak davasını...,legal_qa,OrionCAF/turkish_law_qa_dataset,train,1
2,orioncaf_2,"Soru: Gerçek malik tapu sicilinde görünüyorsa,...","Gerçek malik tapu sicilinde görünüyorsa, hangi...","Gerçek malik tapu sicilinde görünüyorsa, istih...",legal_qa,OrionCAF/turkish_law_qa_dataset,train,2


In [50]:
heldout_df = pd.DataFrame(heldout)
heldout_df.head()

,question,gold_passage_id,kind,notes
0,Yabancı para birimi üzerinden ipotek tesis edi...,orioncaf_3696,dataset_title_seed,Auto-created seed item; replace or augment wit...
1,Silah kaçakçılığı ve ticareti suçları için ert...,orioncaf_830,dataset_title_seed,Auto-created seed item; replace or augment wit...
2,Kamu görevlisinin kişisel kusurla hareket etme...,orioncaf_9122,dataset_title_seed,Auto-created seed item; replace or augment wit...
3,Mirasçılığın gizlenmesi durumunda tapu iptal v...,orioncaf_8123,dataset_title_seed,Auto-created seed item; replace or augment wit...
4,TMK madde 175'e göre yoksulluk nafakasının tal...,orioncaf_7408,dataset_title_seed,Auto-created seed item; replace or augment wit...


## Choose Corpus Size

To keep the first runs fast, we start with a subset but make sure the held-out gold passages are included. After this works, we can set `MAX_CORPUS_SIZE = None` and use the full corpus.

In [51]:
MAX_CORPUS_SIZE = 2000  # set to None for full corpus after the baseline works

passage_by_id = {p["passage_id"]: p for p in passages}
gold_ids = {item["gold_passage_id"] for item in heldout if item.get("gold_passage_id")}
gold_passages = [passage_by_id[pid] for pid in gold_ids if pid in passage_by_id]
other_passages = [p for p in passages if p["passage_id"] not in gold_ids]

if MAX_CORPUS_SIZE is None or MAX_CORPUS_SIZE >= len(passages):
    corpus = passages
else:
    remaining = max(0, MAX_CORPUS_SIZE - len(gold_passages))
    sampled_other = random.sample(other_passages, min(remaining, len(other_passages)))
    corpus = gold_passages + sampled_other

corpus_by_id = {p["passage_id"]: p for p in corpus}
missing_gold = sorted(gold_ids - set(corpus_by_id))

print("corpus size:", len(corpus))
print("gold passages included:", len(gold_passages), "/", len(gold_ids))
print("missing gold IDs:", len(missing_gold))

if missing_gold:
    print("\nNote: some held-out gold passages are not present in passages_seed.jsonl.")
    print("This probably means notebook 01 was run before we fixed the held-out logic.")
    print("Clean fix: rerun notebook 01 from 'Create Initial Held-Out Seed' onward.")
    print("For now, using an in-memory eval set sampled from the current corpus.")

    fallback_size = min(50, len(corpus))
    eval_items = [
        {
            "question": p["title"],
            "gold_passage_id": p["passage_id"],
            "kind": "fallback_from_current_corpus",
            "notes": "Temporary in-memory item because heldout_seed gold IDs were missing from corpus.",
        }
        for p in random.sample(corpus, fallback_size)
    ]
else:
    eval_items = heldout

print("eval items for this run:", len(eval_items))

corpus size: 2000
gold passages included: 50 / 50
missing gold IDs: 0
eval items for this run: 50


## Build Embeddings

`intfloat/multilingual-e5-base` is our first multilingual retrieval baseline. E5 models use `passage:` and `query:` prefixes, so we add them before encoding.

In [52]:
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
BATCH_SIZE = 32

model = SentenceTransformer(EMBEDDING_MODEL)
model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15075.98it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [53]:
def prefix_passage(text: str) -> str:
    return f"passage: {text}"


def prefix_query(text: str) -> str:
    return f"query: {text}"


corpus_texts = [prefix_passage(p["text"]) for p in corpus]

passage_embeddings = model.encode(
    corpus_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
passage_embeddings = np.asarray(passage_embeddings, dtype="float32")
passage_embeddings.shape

Batches: 100%|██████████| 63/63 [00:05<00:00, 11.61it/s]


(2000, 768)

## Build Search Index

In [54]:
if HAVE_FAISS:
    dim = passage_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(passage_embeddings)
    print("FAISS index size:", index.ntotal)
else:
    index = None
    print("FAISS not installed; using NumPy cosine search fallback.")

FAISS index size: 2000


In [55]:
def search(query: str, k: int = 5) -> list[dict]:
    q_emb = model.encode(
        [prefix_query(query)],
        normalize_embeddings=True,
    )
    q_emb = np.asarray(q_emb, dtype="float32")

    if HAVE_FAISS:
        scores, ids = index.search(q_emb, k)
        pairs = zip(scores[0].tolist(), ids[0].tolist())
    else:
        sims = passage_embeddings @ q_emb[0]
        top_ids = np.argsort(-sims)[:k]
        pairs = [(float(sims[i]), int(i)) for i in top_ids]

    results = []
    for rank, (score, idx) in enumerate(pairs, start=1):
        if idx < 0:
            continue
        p = corpus[idx]
        results.append(
            {
                "rank": rank,
                "score": float(score),
                "passage_id": p["passage_id"],
                "title": p["title"],
                "snippet": p["snippet"],
                "tag": p.get("tag", ""),
            }
        )
    return results

## Manual Retrieval Checks

These examples are for a quick manual check. The question is whether the retrieved passages are actually relevant, not only similar-looking text.

In [56]:
manual_queries = [
    "Kira sözleşmesi bitmeden kiracı evden çıkabilir mi?",
    "Boşanma davasında nafaka nasıl belirlenir?",
    "Miras kalan evin paylaşımı için ne yapılabilir?",
    "İşten çıkarılan işçi kıdem tazminatı alabilir mi?",
    "Komşum gürültü yapıyor, hemen dava açmalı mıyım?",
]

for query in manual_queries:
    print("\nQUERY:", query)
    for hit in search(query, k=3):
        print(f"  #{hit['rank']} score={hit['score']:.3f} id={hit['passage_id']}")
        print("   ", hit["title"][:180])


QUERY: Kira sözleşmesi bitmeden kiracı evden çıkabilir mi?
  #1 score=0.868 id=orioncaf_16240
    Kira sözleşmesinin süresi ne olursa olsun, kiracının tahliyesi için hangi maddeye dayanarak dava açılabilir?
  #2 score=0.868 id=orioncaf_6371
    Kira sözleşmesi sona erdiğinde kiracının hangi borçlarını yerine getirmesi gerekmektedir?
  #3 score=0.859 id=orioncaf_14969
    Kiracı, işyeri kira sözleşmesinin devri durumunda hangi sorumlulukları taşımaktadır?

QUERY: Boşanma davasında nafaka nasıl belirlenir?
  #1 score=0.882 id=orioncaf_15576
    Tarafların anlaşmalı boşanma davasında iştirak nafakası ile ilgili hangi hususlarda mutabakata varması mümkündür?
  #2 score=0.880 id=orioncaf_15657
    Tedbir nafakasının boşanma veya ayrılık davası ile birlikte talep edilmesi durumunda hangi mahkeme görevli ve yetkilidir?
  #3 score=0.879 id=orioncaf_13128
    Boşanma davası sürecinde veya sonrasında eşlerin nafaka talep etme hakları nelerdir?

QUERY: Miras kalan evin paylaşımı için ne yapılabi

## Retrieval Metrics On Seed Held-Out Set

This is only a first sanity check. The seed questions are close to the dataset wording, so the numbers may look better than they really are. The final evaluation should include paraphrased, ambiguous, unsupported, and legal-advice-risk questions.

In [57]:
DETAIL_COLUMNS = ["question", "gold_passage_id", "rank", "top_1", "top_3"]


def evaluate_retrieval(items: list[dict], ks=(3, 5, 8)) -> tuple[dict, pd.DataFrame]:
    max_k = max(ks)
    usable_items = [item for item in items if item.get("gold_passage_id") in corpus_by_id]
    rows = []
    recall_hits = {k: 0 for k in ks}
    reciprocal_ranks = []

    for item in tqdm(usable_items):
        query = item["question"]
        gold_id = item["gold_passage_id"]
        hits = search(query, k=max_k)
        hit_ids = [h["passage_id"] for h in hits]

        rank = None
        if gold_id in hit_ids:
            rank = hit_ids.index(gold_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

        for k in ks:
            if gold_id in hit_ids[:k]:
                recall_hits[k] += 1

        rows.append(
            {
                "question": query,
                "gold_passage_id": gold_id,
                "rank": rank,
                "top_1": hit_ids[0] if hit_ids else None,
                "top_3": hit_ids[:3],
            }
        )

    n = len(usable_items)
    metrics = {
        "num_eval_items": n,
        "embedding_model": EMBEDDING_MODEL,
        "corpus_size": len(corpus),
        "mrr": float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0,
    }
    for k in ks:
        metrics[f"recall_at_{k}"] = recall_hits[k] / n if n else 0.0

    return metrics, pd.DataFrame(rows, columns=DETAIL_COLUMNS)


metrics, detail_df = evaluate_retrieval(eval_items)
metrics

100%|██████████| 50/50 [00:00<00:00, 76.05it/s]


{'num_eval_items': 50,
 'embedding_model': 'intfloat/multilingual-e5-base',
 'corpus_size': 2000,
 'mrr': 0.99,
 'recall_at_3': 1.0,
 'recall_at_5': 1.0,
 'recall_at_8': 1.0}

In [58]:
if detail_df.empty:
    print("No usable evaluation rows. Re-run notebook 01 so heldout gold IDs exist in passages_seed.jsonl.")
else:
    display(detail_df.sort_values(["rank"], na_position="first").head(10))

,question,gold_passage_id,rank,top_1,top_3
0,Yabancı para birimi üzerinden ipotek tesis edi...,orioncaf_3696,1,orioncaf_3696,"[orioncaf_3696, orioncaf_3695, orioncaf_12112]"
27,İdari işlemden kaynaklanan istirdat davalarınd...,orioncaf_9215,1,orioncaf_9215,"[orioncaf_9215, orioncaf_9201, orioncaf_8260]"
28,Uzun dönem oturma izni alan bir yabancı ne gib...,orioncaf_5169,1,orioncaf_5169,"[orioncaf_5169, orioncaf_5188, orioncaf_5092]"
29,Gecikmiş itiraz nedir ve hangi durumlarda başv...,orioncaf_7148,1,orioncaf_7148,"[orioncaf_7148, orioncaf_7146, orioncaf_8660]"
30,Nüfus Hizmetleri Kanunu'na eklenen 27/A maddes...,orioncaf_11154,1,orioncaf_11154,"[orioncaf_11154, orioncaf_11165, orioncaf_17226]"
31,"Defter, kayıt ve belgeleri tahrif edenler veya...",orioncaf_3392,1,orioncaf_3392,"[orioncaf_3392, orioncaf_3396, orioncaf_10146]"
32,Arabuluculuk sürecinde tarafların temsil şekil...,orioncaf_3082,1,orioncaf_3082,"[orioncaf_3082, orioncaf_5660, orioncaf_3006]"
33,Cari hesaba genellikle hangi tür alacaklar geç...,orioncaf_12595,1,orioncaf_12595,"[orioncaf_12595, orioncaf_17251, orioncaf_2685]"
34,Anonim şirketlerde iş kazaları sonucu yönetim ...,orioncaf_3212,1,orioncaf_3212,"[orioncaf_3212, orioncaf_3232, orioncaf_3170]"
35,İstinafa başvurma hakkından zımni olarak ferag...,orioncaf_11899,1,orioncaf_11899,"[orioncaf_11899, orioncaf_9094, orioncaf_8904]"


## Error Inspection

Here we look at cases where the gold passage is missing from the top results. These examples can tell us whether we need better cleaning, another embedding model, query rewriting, or reranking.

In [59]:
misses = detail_df[detail_df["rank"].isna()].copy()
print("misses:", len(misses))
misses.head(10)

misses: 0


,question,gold_passage_id,rank,top_1,top_3


In [60]:
if len(misses):
    row = misses.iloc[0]
    print("QUESTION:", row["question"])
    print("GOLD:", row["gold_passage_id"])
    print("\nTOP RESULTS")
    for hit in search(row["question"], k=5):
        print(f"#{hit['rank']} {hit['passage_id']} score={hit['score']:.3f}")
        print(hit["title"][:250])
        print()

## Save Baseline Results

In [61]:
metrics_path = RESULTS_DIR / "retrieval_baseline_seed_metrics.json"
detail_path = RESULTS_DIR / "retrieval_baseline_seed_details.csv"

metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
detail_df.to_csv(detail_path, index=False)

print("wrote", metrics_path)
print("wrote", detail_path)

wrote /Users/onur/Desktop/CS455/Term Project/evaluation/results/retrieval_baseline_seed_metrics.json
wrote /Users/onur/Desktop/CS455/Term Project/evaluation/results/retrieval_baseline_seed_details.csv


## Notes After Running

After running this notebook, we should write down:

- Does the gold passage usually appear in top-3/top-5?
- Are top results useful even when the gold ID is missed?
- Which topics retrieve poorly?
- Are failures caused by vague questions, short answers, duplicates, or weak embeddings?
- What should be tested next: larger corpus, different embedding model, reranker, or query rewriting?

These notes can later become part of the retrieval and error analysis sections of the final report.